In [2]:
# figure4.py
# Figure 4 (EGD) and Figure S2 (UTSW): decision regions of all six decision rules on one representative split (90% target).
# Landscape layout: 2 rows x 3 columns. Columns match the rule types of Table 1: single-label rules, probability-based rules,
# conformal prediction. The two rules of each type are stacked in that column (A/B, C/D, E/F).
# Runs on its own. Reads modeling_table.csv and, from core_outputs/ (1_code.ipynb):
#   egd_probas.npy, utsw_probas.npy, is_test_egd.npy, is_test_utsw.npy, decision_rules_200_splits.csv
# Writes core_outputs/Figure4.png/.tiff and FigureS2.png/.tiff
# Each patient's three probabilities sum to 1, so every patient is a point in a triangle.
# Corners: glioblastoma bottom left, astrocytoma bottom right, oligodendroglioma top. A probability is read as the
# distance from the side opposite its corner (gridlines at 0.2 to 0.8, labelled in the subtype colour).
# The background is the set a rule returns at that point. Markers are the test-half patients.
# All rules are fitted on the calibration half of this split, exactly as in 1_code.ipynb.

import os
import numpy as np
import pandas as pd
from itertools import product
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT = 'core_outputs'
table = pd.read_csv('modeling_table.csv')
y_egd = table[table['site'] == 'EGD']['2021_glioma_class'].values
y_utsw = table[table['site'] == 'UTSW']['2021_glioma_class'].values
classes = np.unique(table[table['site'].isin(['TCGA', 'UCSF'])]['2021_glioma_class'].values)
cohorts = {'EGD': (y_egd, np.load(os.path.join(OUT, 'egd_probas.npy'))),
           'UTSW': (y_utsw, np.load(os.path.join(OUT, 'utsw_probas.npy')))}
is_test = {'EGD': np.load(os.path.join(OUT, 'is_test_egd.npy')), 'UTSW': np.load(os.path.join(OUT, 'is_test_utsw.npy'))}
rules = pd.read_csv(os.path.join(OUT, 'decision_rules_200_splits.csv'))
eye3 = np.eye(3, dtype=int)
weight_grid = [np.array(w, dtype=float) for w in product([0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0], repeat=3)]   # same grid as 1_code.ipynb

plt.rcParams.update({'font.family': ['Arial', 'Liberation Sans', 'DejaVu Sans'], 'font.size': 7.5})

# split closest to the 200-split means across all six rules, both cohorts (coverage per subtype and set size)
cols = ['cov_A', 'cov_G', 'cov_O', 'size']
dist = rules[cols] - rules.groupby(['cohort', 'rule'])[cols].transform('mean')
seed = int((dist ** 2).sum(axis=1).groupby(rules['seed']).sum().idxmin())
print('split used for Figure 4:', seed)
level = 0.90
h = np.sqrt(3) / 2                                   # triangle height

# grid of probability triples covering the triangle (one per pixel), columns in class order A, G, O
px, py = np.meshgrid(np.linspace(0, 1, 900), np.linspace(0, h, 780))
p_o = py / h
p_a = px - 0.5 * p_o
p_g = 1 - p_a - p_o
inside = (p_a >= 0) & (p_g >= 0) & (p_o >= 0)
grid = np.column_stack([p_a[inside], p_g[inside], p_o[inside]])

# colours: light tint of each subtype colour, pairs are blends, all three is gray. Set code = 1*A + 2*G + 4*O
sub_colors = ['#0072B2', '#D55E00', '#009E73']        # A, G, O
tint = [0.30 * np.array(matplotlib.colors.to_rgb(c)) + 0.70 for c in sub_colors]
set_rgb = np.clip(np.array([[1, 1, 1], tint[0], tint[1], (tint[0] + tint[1]) / 2 - 0.05, tint[2],
                            (tint[0] + tint[2]) / 2 - 0.05, (tint[1] + tint[2]) / 2 - 0.05, [0.86, 0.86, 0.86]]), 0, 1)
set_names = ['empty', '{A}', '{G}', '{A, G}', '{O}', '{A, O}', '{G, O}', 'all 3']
marks = ['o', 's', '^']

for cohort, out_name in [('EGD', 'Figure4'), ('UTSW', 'FigureS2')]:
    y, p = cohorts[cohort]
    true_col = np.searchsorted(classes, y)
    test = np.where(is_test[cohort][seed])[0]
    cal = np.where(~is_test[cohort][seed])[0]
    yt = true_col[test]
    cal_scores = 1 - p[cal, true_col[cal]]

    # every rule is applied to the grid and the test patients together (test patients are the last rows)
    pts = np.vstack([grid, p[test]])

    # recalibrated probabilities (rule 1)
    pts_clip = np.clip(pts, 1e-6, 1 - 1e-6)
    p_clip = np.clip(p, 1e-6, 1 - 1e-6)
    recal = np.zeros(pts.shape)
    for k in range(3):
        sig = LogisticRegression(C=1e6, max_iter=1000).fit(np.log(p_clip[cal, k] / (1 - p_clip[cal, k])).reshape(-1, 1), (true_col[cal] == k).astype(int))
        recal[:, k] = sig.predict_proba(np.log(pts_clip[:, k] / (1 - pts_clip[:, k])).reshape(-1, 1))[:, 1]
    recal = recal / recal.sum(axis=1, keepdims=True)
    set_recal_argmax = eye3[np.argmax(recal, axis=1)]

    # class-weighted argmax (rule 2): first weight vector with the best balanced accuracy on the calibration half
    bal = [balanced_accuracy_score(true_col[cal], np.argmax(p[cal] * w, axis=1)) for w in weight_grid]
    best_w = weight_grid[int(np.argmax(bal))]
    set_weighted = eye3[np.argmax(pts * best_w, axis=1)]
    print(cohort, 'class weights on this calibration half (A, G, O):', best_w)

    # probability sets (rule 3)
    order = np.argsort(-recal, axis=1)
    sorted_p = np.take_along_axis(recal, order, axis=1)
    set_prob = np.zeros(pts.shape, dtype=int)
    np.put_along_axis(set_prob, order, ((np.cumsum(sorted_p, axis=1) - sorted_p) < level).astype(int), axis=1)

    # conformal cutoffs (rules 5 and 6): a subtype is listed when its probability >= 1 - (largest passing calibration score).
    # A score passes when (number of calibration scores >= it, plus 1) / (n + 1) >= 1 - level. Same rule as crepes.
    cut_single = 1 - max(s for s in cal_scores if (np.sum(cal_scores >= s) + 1) / (len(cal_scores) + 1) >= 1 - level)
    cut_mondrian = []
    for k in range(3):
        sc = cal_scores[true_col[cal] == k]
        cut_mondrian.append(1 - max(s for s in sc if (np.sum(sc >= s) + 1) / (len(sc) + 1) >= 1 - level))

    # abstention (rule 4): commit share = Mondrian single-label share on the calibration half
    n_commit = int(round(np.mean((p[cal] >= np.array(cut_mondrian)).sum(axis=1) == 1) * len(cal)))
    cut_abstain = np.sort(p[cal].max(axis=1))[::-1][n_commit - 1]
    set_abstain = np.ones(pts.shape, dtype=int)
    commit = pts.max(axis=1) >= cut_abstain
    set_abstain[commit] = eye3[np.argmax(pts, axis=1)][commit]

    groups = ['Single-label rules', 'Probability-based rules', 'Conformal prediction']
    panels = [('A', 'Recalibrated argmax', set_recal_argmax, []),
              ('B', 'Class-weighted argmax', set_weighted, []),
              ('C', 'Probability sets', set_prob, []),
              ('D', 'Probability-based abstention', set_abstain, []),
              ('E', 'Standard conformal', (pts >= cut_single).astype(int), [cut_single] * 3),
              ('F', 'Class-conditional conformal', (pts >= np.array(cut_mondrian)).astype(int), cut_mondrian)]

    # landscape: 2 rows x 3 columns, one column per rule type, the two rules of a type stacked top and bottom
    fig, axes = plt.subplots(2, 3, figsize=(10.4, 7.0))
    panel_axes = [axes[0, 0], axes[1, 0], axes[0, 1], axes[1, 1], axes[0, 2], axes[1, 2]]
    for ax, (letter, title, sets, cutoffs) in zip(panel_axes, panels):
        code = sets[:, 0] + 2 * sets[:, 1] + 4 * sets[:, 2]
        grid_code = code[:len(grid)]
        test_sets = sets[len(grid):]

        # background: set colour inside the triangle, thin dark lines where the set changes, white outside
        code_img = np.full(px.shape, -1)
        code_img[inside] = grid_code
        edge = np.zeros(px.shape, dtype=bool)
        edge[:, 1:] = code_img[:, 1:] != code_img[:, :-1]
        edge[1:, :] = edge[1:, :] | (code_img[1:, :] != code_img[:-1, :])
        image = np.ones(px.shape + (3,))
        image[inside] = set_rgb[grid_code]
        image[edge & inside] = 0.45
        ax.imshow(image, origin='lower', extent=[0, 1, 0, h], interpolation='nearest')

        # gridlines at 0.2 to 0.8 for each probability, labelled on one side in the subtype colour
        for k in range(3):
            for v in [0.2, 0.4, 0.6, 0.8]:
                e1 = np.roll([v, 1 - v, 0.0], k)
                e2 = np.roll([v, 0.0, 1 - v], k)
                ax.plot([e1[0] + 0.5 * e1[2], e2[0] + 0.5 * e2[2]], [e1[2] * h, e2[2] * h], color='white', linewidth=0.5, alpha=0.8)
                lab = e2                                               # end on A's right side, G's bottom side, O's left side
                dx = [0.035, 0.0, -0.035][k]                           # push the label just outside that side
                dy = [0.02, -0.035, 0.02][k]
                ax.text(lab[0] + 0.5 * lab[2] + dx, lab[2] * h + dy, '{:.1f}'.format(v), color=sub_colors[k], fontsize=6, ha='center', va='center')

        # region names at their centres, if large enough
        for k in np.unique(grid_code):
            region = grid_code == k
            if region.mean() >= 0.03:
                ax.text(px[inside][region].mean(), py[inside][region].mean(), set_names[k], ha='center', va='center', fontsize=7.5,
                        fontweight='bold', bbox=dict(boxstyle='round,pad=0.12', facecolor='white', edgecolor='none', alpha=0.8))

        # conformal cutoffs: dashed line where that subtype's probability equals its cutoff
        for k, c in enumerate(cutoffs):
            e1 = np.roll([c, 1 - c, 0.0], k)
            e2 = np.roll([c, 0.0, 1 - c], k)
            ax.plot([e1[0] + 0.5 * e1[2], e2[0] + 0.5 * e2[2]], [e1[2] * h, e2[2] * h], color=sub_colors[k], linestyle='--', linewidth=1.1)

        # test-half patients by true subtype
        for k in range(3):
            d = p[test][yt == k]
            ax.scatter(d[:, 0] + 0.5 * d[:, 2], d[:, 2] * h, s=6, marker=marks[k], color=sub_colors[k], edgecolors='black', linewidths=0.25, zorder=3)

        # triangle outline and corner names
        ax.plot([0, 1, 0.5, 0], [0, 0, h, 0], color='black', linewidth=0.8)
        ax.text(-0.02, -0.08, 'Glioblastoma', ha='center', va='top', color=sub_colors[1], fontweight='bold')
        ax.text(1.02, -0.08, 'Astrocytoma', ha='center', va='top', color=sub_colors[0], fontweight='bold')
        ax.text(0.5, h + 0.03, 'Oligodendroglioma', ha='center', va='bottom', color=sub_colors[2], fontweight='bold')

        # this split: coverage per subtype and mean set size
        covered = test_sets[np.arange(len(test)), yt] == 1
        ax.text(0.5, -0.17, 'Coverage A {:.2f}, G {:.2f}, O {:.2f}   Mean set size {:.2f}'.format(
            covered[yt == 0].mean(), covered[yt == 1].mean(), covered[yt == 2].mean(), test_sets.sum(axis=1).mean()),
            ha='center', va='top', fontsize=7)
        ax.set_title(letter + '   ' + title, loc='left', fontweight='bold', fontsize=8.5)
        ax.set_xlim(-0.1, 1.1)
        ax.set_ylim(-0.24, h + 0.08)
        ax.set_aspect('equal')
        ax.axis('off')

    handles = [Line2D([0], [0], marker=marks[k], color='none', markerfacecolor=sub_colors[k], markeredgecolor='black',
                      markeredgewidth=0.25, markersize=5, label=classes[k].capitalize() + ' (true subtype)') for k in range(3)]
    handles.append(Line2D([0], [0], color='#555555', linestyle='--', linewidth=1.1, label='Conformal cutoff'))
    fig.legend(handles=handles, loc='lower center', ncol=4, frameon=False, fontsize=7.5)
    plt.tight_layout(rect=[0, 0.035, 1, 0.94], h_pad=2.0, w_pad=1.5)

    # column labels, centred above the top panel of each column
    for col in range(3):
        box = axes[0, col].get_position()
        fig.text((box.x0 + box.x1) / 2, box.y1 + 0.035, groups[col], fontsize=10, fontweight='bold', color='#444444', ha='center', va='bottom')
    plt.savefig(os.path.join(OUT, out_name + '.png'), dpi=300)
    plt.savefig(os.path.join(OUT, out_name + '.tiff'), dpi=600, pil_kwargs={'compression': 'tiff_lzw'})
    plt.close(fig)

split used for Figure 4: 87
EGD class weights on this calibration half (A, G, O): [1.  0.5 8. ]


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati

UTSW class weights on this calibration half (A, G, O): [2.   0.75 8.  ]


findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberation Sans' not found.
findfont: Font family 'Liberati